# Shelf List Report

This notebook will create a CSV file of Inventory Items with the following information: Barcode, Title, Effective Location, Effective Call Number Components, Material Type, and Item Status

## 1. Environment setup

In [ ]:
# This script will import the following Python libraries, but not install them. If you are missing any of these, you can install them via the command line like this:
# !pip install pandas
import pandas as pd
import requests
from datetime import datetime, timedelta   

pd.set_option('display.max_columns', None)

## 2. Log in from a shared notebook

This is the notebook that will authenticate you to the FOLIO API and store your access token in a variable called ACCESS_TOKEN. You will need to run this cell before running any other cells in this notebook.

In [ ]:
%run folio_auth.ipynb

## 3. Helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. 

In [ ]:

def fetch_all_records(endpoint, records_key, limit=1000, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit

    return all_records

## 4. Get Locations

In [ ]:


locations_raw = fetch_all_records(
    "/locations",
    records_key="locations",
    query='isActive=="true"',
) 

print(f"{len(locations_raw)} active locations found")
locations_df = pd.DataFrame(locations_raw)
print(locations_df.loc[:, ['name','id']])


## 5. Get # of records to process for one location

In [60]:
location_id = 'fc4b17f0-3745-4827-8e74-08f004003f7d'
# location_id = 'REPLACE WITH THE LOCATION ID FROM ABOVE'

items_raw = fetch_all_records(
    "/item-storage/items",
    records_key="items",
    query='limit=0&query=(effectiveLocationId=='+location_id+')'
) 
location_name = locations_df[locations_df['id']==location_id]['name'].values[0]
print(f"{len(items_raw)} items found for with an effective location of: {location_name}")


10 items found for with an effective location of: MED Director's Office


## 6. Retrieve Item records with that Effective Location